In [ ]:
# import json

In [ ]:
# def load_jsonl(filename):
#     data = []
#     with open(filename, 'r', encoding='utf-8') as f:
#         for line in f:
#             try:
#                 # Use json.loads() to deserialize each line string into a Python dictionary
#                 json_object = json.loads(line.strip())
#                 data.append(json_object)
#             except json.JSONDecodeError as e:
#                 print(f"Error decoding JSON on line: {line.strip()}. Error: {e}")
#     return data

In [ ]:
# dataset = load_jsonl('dataset.jsonl')
# print(len(dataset))
# print(dataset[0])

In [ ]:
# question_list = []

In [ ]:
# dataset = pd.read_csv('dataset.csv')
# dataset.head()

In [1]:
# Get Data from Redis
import os
import json
import redis
import random
import pandas as pd
from redisvl.index import SearchIndex
from redis.commands.search.query import Query

# from galileo import galileo_context
# from galileo.openai import openai
from openai import OpenAI
# from galileo.config import GalileoPythonConfig
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv

In [2]:
# load env vars from .env file
load_dotenv()

True

In [3]:
REDIS_URL = f"redis://default:{os.getenv('REDIS_PASSWORD')}@{os.getenv('REDIS_HOST')}:{os.getenv('REDIS_PORT')}"
r = redis.from_url(REDIS_URL, decode_responses=True)

if r.ping():
    print("Connection successful!")
else:
    print("Connection issue!")

Connection successful!


In [4]:
index_name = "finance-local-index"
index = SearchIndex.from_existing(index_name, r)

In [5]:
def get_index_status():
  info = r.ft(index_name).info()
  print(info['percent_indexed'])
  print(info['num_records'])
  return info

In [6]:
index_info = get_index_status()

1
6191


In [7]:
def search_by_metadata(company):
    query = Query(f'@_metadata_json:{company}').paging(0,1)
    return r.ft(index_name).search(query).docs

In [8]:
company = "Nvidia"
doc = search_by_metadata(company)
doc[0].text

'Financial results for NVIDIA\nNVIDIA delivered record-breaking financial performance in the third quarter of fiscal 2025, with both revenue and profit reaching historic highs. Total revenue for the quarter was $35.1 billion, reflecting a 17% sequential increase from the previous quarter and an impressive 94% year-over-year growth, notably surpassing the company’s projected outlook of $32.5 billion. Data center revenue was the primary growth engine, accounting for $30.8 billion, up 17% quarter-over-quarter and 112% compared to the same period last year. This surge is attributed to expanding adoption of NVIDIA accelerated computing and AI infrastructure across cloud service providers and enterprises. Networking revenue also demonstrated robust performance, rising 20% year-over-year, with particular strength in segments such as InfiniBand, Ethernet switches, SmartNICs, BlueField DPUs, and a threefold increase in Spectrum-X Ethernet for AI revenues.\nOther business segments also contribut

In [ ]:
list_of_companies = ["Nvidia", "HomeDepot", "Google", "Target", "Intuit", "Amazon", "eBay", "Broadcom", "GAP", "Apple", "Disney", "Walmart",
                     "Oracle", "Costco", "UltaBeauty", "Meta", "Microsoft", "Adobe", "Tesla", "NetApp", "Salesforce"]
len(list_of_companies)

In [ ]:
MOCK_PRICE_DB = {
    "Costco": {"price": 999.63, "change": -8.80, "change_percent": -0.87, "volume": 502, "high": 1012.65, "low": 95.43, "open": 1010.50, "company_name": "Costco Wholesale Corp", "currency": "USD" },
    "Adobe": {"price": 254.55, "change": 5.26, "change_percent": 2.11, "volume": 502, "high": 255.00, "low": 244.55, "open": 248.25, "company_name": "Adobe Inc", "currency": "USD"},
    "Oracle": {"price": 156.61, "change": 1.50, "change_percent": 1.03, "volume": 502, "high": 158.74, "low": 155.15, "open": 156.06, "company_name": "Oracle Corp", "currency": "USD"},
    "NetApp": {"price": 102.90, "change": 3.43, "change_percent": 3.48, "volume": 502, "high": 102.29, "low": 99.76, "open": 100.36, "company_name": "NetApp Inc", "currency": "USD"},
    "Walmart":  {"price": 125.37, "change": -1.11, "change_percent": -0.88, "volume": 502, "high": 126.98, "low": 124.83, "open": 126.76, "company_name": "Walmart Inc", "currency": "USD"},
    "Disney":  {"price": 98.94, "change": -0.35, "change_percent": -0.35, "volume": 502, "high": 99.69, "low": 98.53, "open": 99.47, "company_name": "Walt Disney Co", "currency": "USD"},
    "eBay": {"price": 90.95, "change": -0.39, "change_percent": -0.34, "volume": 502, "high": 92.50, "low": 90.70, "open": 91.91, "company_name": "eBay Inc", "currency": "USD"},
    "Target":  {"price": 117.30, "change": -0.050, "change_percent": -0.043, "volume": 502, "high": 118.45, "low": 116.00, "open": 118.00, "company_name": "Target Corp", "currency": "USD"},
    "HomeDepot":   {"price": 342.75, "change": 3.64, "change_percent": 1.07, "volume": 502, "high": 345.24, "low": 340.65, "open": 342.44, "company_name": "Home Depot Inc", "currency": "USD" },
    "Broadcom": {"price": 184.72, "change": -2.68, "change_percent": -1.43, "volume": 502, "high": 186.34, "low": 184.52, "open": 186.24, "company_name": "Broadcom Inc.", "currency": "USD"},
    "UltaBeauty": {"price": 532.41, "change": -3.31, "change_percent": -0.62, "volume": 502, "high": 542.84, "low": 530.85, "open": 532.92, "company_name": "Ulta Beauty Inc", "currency": "USD"},
    "Intuit": {"price": 453.51, "change": 13.55, "change_percent": 2.96, "volume": 502, "high": 454.21, "low": 442.11, "open": 445.32, "company_name": "Intuit Inc", "currency": "USD"},
    "GAP":  {"price": 19.85, "change": 0.15, "change_percent": 0.76, "volume": 4567890, "high": 20.00, "low": 19.50, "open": 19.70, "company_name": "Gap Inc.", "currency": "USD"},
    "Apple": {"price": 178.72, "change": 1.23, "change_percent": 0.69, "volume": 52345678, "high": 179.50, "low": 177.80, "open": 178.00, "company_name": "Apple Inc.", "currency": "USD"},
    "Microsoft": {"price": 415.32, "change": 2.45, "change_percent": 0.59, "volume": 23456789, "high": 416.00, "low": 413.50, "open": 414.00, "company_name": "Microsoft Corporation", "currency": "USD"},
    "Google":{"price": 147.68, "change": -0.82, "change_percent": -0.55, "volume": 34567890, "high": 148.50, "low": 147.20, "open": 147.90, "company_name": "Alphabet Inc.", "currency": "USD"},
    "Amazon": {"price": 178.75, "change": 1.25, "change_percent": 0.70, "volume": 45678901, "high": 179.00, "low": 177.50, "open": 178.00, "company_name": "Amazon.com Inc.", "currency": "USD"},
    "Meta": {"price": 485.58, "change": 3.42, "change_percent": 0.71, "volume": 56789012, "high": 486.00, "low": 482.00, "open": 483.00, "company_name": "Meta Platforms Inc.", "currency": "USD"},
    "Tesla": {"price": 177.77, "change": -2.33, "change_percent": -1.29, "volume": 67890123, "high": 180.00, "low": 177.00, "open": 179.00, "company_name": "Tesla Inc.", "currency": "USD"},
    "Nvidia": {"price": 950.02, "change": 15.98, "change_percent": 1.71, "volume": 78901234, "high": 952.00, "low": 945.00, "open": 946.00, "company_name": "NVIDIA Corporation", "currency": "USD"},
    "Salesforce": {"price": 194.34, "change": -0.97, "change_percent": -0.50, "volume": 78901234, "high": 196.15, "low": 192.64, "open": 193.40, "company_name": "Salesforce", "currency": "USD"}
}

In [ ]:
list_of_questions = [
    "What was COMPANY's revenue in the quarter, and how does that compare to previous quarters?",
    "What was COMPANY's financial results?",
    "What can you tell me about COMPANY's results in the quarter, in terms of revenue and profit?",
    "Tell me about COMPANY's revenue trend in the quarter",
    "What was COMPANY's revenue and profit in the quarter?",
    "What were COMPANY's financial results?"
]

In [ ]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [ ]:
def ask_llm(query, context):

    # Get LLM Response
    system_template = """
    You are a financial expert.
    Your task is to answer customer questions by using a given context.
    Keep your answer in line with the context provided.
    Do not start the answer with a comment like 'based on the context provided'.
    %CONTEXT%
    {context}

    """
    messages = [
        {"role": "system", "content": system_template.format(context=context)},
        {"role": "user", "content": query}
    ]

    llm_response = client.chat.completions.create(
        model="gpt-4.1",
        messages=messages,
    ).choices[0].message.content

    return llm_response

In [ ]:
dataset = []
for company in list_of_companies:
# for i in range(1):
    # company = list_of_companies[i]
    doc = search_by_metadata(company)
    text = doc[0].text
    # stock = MOCK_PRICE_DB[company]
    # stock['earnings'] = text
    question = random.choice(list_of_questions).replace("COMPANY", company)
    answer = ask_llm(question, text)
    new_doc = {
        "input": json.dumps({"question": question, "rag_content": text}),
        "ground_truth": answer,
        "generated_output": "",
        "metadata": {"company": company}
    }
    dataset.append(new_doc)


In [ ]:
dataset[0]

In [ ]:
# Dataset must be: Input, Ground Truth, Metadata
df = pd.DataFrame(dataset)
df.head()

In [ ]:
df.to_csv('dataset_finance.csv')

In [ ]:
dataset[0]

In [ ]:
from galileo import Message, MessageRole
from galileo.prompts import create_prompt

from galileo import galileo_context
from galileo.openai import openai
from galileo.config import GalileoPythonConfig
from galileo import GalileoLogger

In [ ]:
os.environ["GALILEO_CONSOLE_URL"] = "https://console.demo-v2.galileocloud.io"
logger = GalileoLogger()
session_id = logger.start_session(name="Test1")
galileo_context.init(project="denisd-project", log_stream="demo-finance")
galileo_context.set_session(session_id)

In [ ]:
# Create a prompt with system and user messages

system_template = """
You are a financial expert.
Your task is to answer customer questions by using a given context.
Keep your answer in line with the context provided.
Do not start the answer with a comment like 'based on the context provided'.
%CONTEXT%
{context}
"""

prompt = create_prompt(
    name="finance-domain-rag-prompt",
    template=[
        Message(role=MessageRole.system,
                content=system_template.format(context=dataset[0]['input']['rag_content'])),
        Message(role=MessageRole.user,
                content=dataset[0]['input']['question'])
    ],
    project_name="denisd-project"
)

In [ ]:
## REDUCE SIZE OF DATASET - TOO MANY TOKENS

In [9]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [10]:
big_df = pd.read_csv('SP500_transcripts_big.csv', index_col=0)
big_df.head()

,Company,Quarter,Transcript
0,Costco,Q1,Richard A. Galanti\nExecutive Vice President a...
1,Adobe,Q4,Shantanu Narayen\nChair & Chief Executive Offi...
2,Oracle,Q2,Safra Catz\nChief Executive Officer at Oracle\...
3,Salesforce,Q1,Marc Benioff\nChief Executive Officer at Sales...
4,NetApp,Q2,George Kurian\nChief Executive Officer and Dir...


In [11]:
transcript_list = big_df['Transcript'].to_list()
company_list = big_df['Company'].to_list()
quarter_list = big_df['Quarter'].to_list()

In [12]:
def ask_llm(query, context):

    # Get LLM Response
    system_template = """
    You are a financial expert.
    Your task is to summarize the provided context, focusing on profit and revenue information contained in the context.
    Do not make up information. Try to keep the output at 5000 characters or less, but do not discard important information about revenue and profit.
    Always include the name of the company in the beginning of the output, like 'Financial results for <company?'. The company name will be in the user's prompt.
    %CONTEXT%
    {context}

    """
    messages = [
        {"role": "system", "content": system_template.format(context=context)},
        {"role": "user", "content": "Please summarize this financial earnings call transcript for company {query}"}
    ]

    llm_response = client.chat.completions.create(
        model="gpt-4.1",
        messages=messages,
    ).choices[0].message.content

    return llm_response

In [13]:
print(len(transcript_list))
print(len(company_list))
print(len(quarter_list))

21
21
21


In [ ]:
dataset = []
for i in range(len(transcript_list)):
    transcript = transcript_list[i]
    company = company_list[i]
    summary = ask_llm(company, transcript)
    new_doc = {
        "Company": company,
        "Quarter": quarter_list[i],
        "Transcript": summary
    }
    dataset.append(new_doc)


In [ ]:
new_df = pd.DataFrame(dataset)
new_df.head()

In [ ]:
# Save to new CSV
new_df.to_csv('SP500_transcripts.csv')

In [ ]:
### STILL TOO BIG - SECOND ROUND OF OPTIMIZATIOn

In [ ]:
still_big_df = pd.read_csv('SP500_transcripts.csv', index_col=0)
still_big_df.head()

In [ ]:
still_big_transcript_list = still_big_df['Transcript'].to_list()
still_big_company_list = still_big_df['Company'].to_list()
still_big_quarter_list = still_big_df['Quarter'].to_list()

In [ ]:
new_df = pd.DataFrame(dataset)
new_df.head()

In [ ]:
# Save to new CSV
new_df.to_csv('SP500_transcripts.csv')

In [ ]:
## STILL TOO BIG - SOLUTION: REDUCE FROM 21 to 16 COMPANIES

In [16]:
df = pd.read_csv('SP500_transcripts_21.csv', index_col=0)
df.head()

,Company,Quarter,Transcript
0,Costco,Q1,Financial results for Costco Wholesale\nFor th...
1,Adobe,Q4,Financial results for Adobe\nIn fiscal year 20...
2,Oracle,Q2,Financial results for Oracle\nOracle delivered...
3,Salesforce,Q1,Financial results for Salesforce:\nSalesforce ...
4,NetApp,Q2,Financial results for NetApp\nNetApp’s financi...


In [17]:
companies_to_delete = ["Disney", "eBay", "UltaBeauty", "Target", "GAP"]

In [18]:
new_dataset = []
for index, row in df.iterrows():
    company = row['Company']
    print(f"{index}: Company: {company}")
    if company not in companies_to_delete:
        new_dataset.append({
            "Company": company,
            "Quarter": row["Quarter"],
            "Transcript": row["Transcript"]
        })

0: Company: Costco
1: Company: Adobe
2: Company: Oracle
3: Company: Salesforce
4: Company: NetApp
5: Company: Walmart
6: Company: Disney
7: Company: eBay
8: Company: Target
9: Company: HomeDepot
10: Company: Broadcom
11: Company: UltaBeauty
12: Company: Intuit
13: Company: GAP
14: Company: Apple
15: Company: Microsoft
16: Company: Google
17: Company: Amazon
18: Company: Tesla
19: Company: Meta
20: Company: Nvidia


In [19]:
for item in new_dataset:
    print(item["Company"])

Costco
Adobe
Oracle
Salesforce
NetApp
Walmart
HomeDepot
Broadcom
Intuit
Apple
Microsoft
Google
Amazon
Tesla
Meta
Nvidia


In [ ]:
df = pd.DataFrame(new_dataset)
df.head()

In [20]:
transcript_list = df['Transcript'].to_list()
company_list = df['Company'].to_list()
quarter_list = df['Quarter'].to_list()

In [21]:
def ask_llm(query, context):

    # Get LLM Response
    system_template = """
    You are a financial expert.
    Your task is to summarize the provided context, focusing on profit and revenue information contained in the context.
    The summary needs to focus on the quarter's results. If the fiscal year results are mentioned, they can be ignored. Also, any future plans are irrelevant for the summary.
    Summarize the bullet points for each category into paragraphs. Do not worry about creating a markup output, as this information will be vectorized.
    Minimize the use of line breaks and bullet points, and remove markup notations.
    Do not make up information. Try to keep the output at 5000 characters or less, but do not discard important information about revenue and profit for the quarter.
    Preserve the name of the company in the beginning of the output, like 'Financial Qx results for <company?'. The company name will be in the user's prompt, and Qx represents the quarter.
    %CONTEXT%
    {context}

    """
    messages = [
        {"role": "system", "content": system_template.format(context=context)},
        {"role": "user", "content": "Please summarize this financial earnings call transcript for company {query}"}
    ]

    llm_response = client.chat.completions.create(
        model="gpt-4.1",
        messages=messages,
    ).choices[0].message.content

    return llm_response

In [22]:
dataset = []
for i in range(len(transcript_list)):
    transcript = transcript_list[i]
    company = company_list[i]
    summary = ask_llm(company, transcript)
    new_doc = {
        "Company": company,
        "Quarter": quarter_list[i],
        "Transcript": summary
    }
    dataset.append(new_doc)

In [23]:
# save output
new_df = pd.DataFrame(dataset)
new_df.head()

,Company,Quarter,Transcript
0,Costco,Q1,Financial Q1 results for Costco Wholesale: Cos...
1,Adobe,Q4,Financial Q4 results for Adobe\n\nIn the fourt...
2,Oracle,Q2,Financial Qx results for Oracle:\n\nOracle del...
3,Salesforce,Q1,Financial Q3 results for Salesforce:\n\nSalesf...
4,NetApp,Q2,Financial Q2 results for NetApp\n\nNetApp’s fi...


In [24]:
# Save to new CSV
new_df.to_csv('SP500_transcripts.csv')